In [2]:
import os
import json

# Project Root Directory Setup
PROJECT_ROOT = os.getcwd()
DATA_RAW_DIR = os.path.join(PROJECT_ROOT, "data", "raw")
DATA_PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

PATHS = {
    "raw_dir": DATA_RAW_DIR,
    "processed_dir": DATA_PROCESSED_DIR,
    "metadata": os.path.join(DATA_PROCESSED_DIR, "metadata.json"),
    "chunks_output": os.path.join(DATA_PROCESSED_DIR, "chunks.json")
}

# Verify Metadata Existence
if os.path.exists(PATHS["metadata"]):
    with open(PATHS["metadata"], "r", encoding="utf-8") as f:
        metadata_catalog = json.load(f)
    print(f"metadata.json length =  {len(metadata_catalog)}")
else:
    print("Not found")

metadata.json length =  410


In [3]:
import pandas as pd
import pdfplumber
from docx import Document

def extract_text_from_pdf(pdf_path: str) -> str:
    """Extracts page body text and formats embedded grid tables from PDFs."""
    text_content = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages, 1):
                page_text = page.extract_text()
                if page_text:
                    text_content.append(f"--- Page {page_num} ---\n{page_text}")

                # Extract embedded tables
                tables = page.extract_tables()
                for table in tables:
                    table_str = "\n".join([" | ".join([str(cell or "").strip() for cell in row]) for row in table if any(row)])
                    if table_str:
                        text_content.append(f"\n[TABLE Page {page_num}]\n{table_str}\n")
    except Exception as e:
        print(f"   [ERROR PDF] {pdf_path}: {e}")
    return "\n\n".join(text_content)

def extract_text_from_docx(docx_path: str) -> str:
    """Extracts paragraphs and tables from Microsoft Word (.docx) schedule files."""
    text_content = []
    try:
        doc = Document(docx_path)
        for p in doc.paragraphs:
            if p.text.strip():
                text_content.append(p.text.strip())
        for table in doc.tables:
            for row in table.rows:
                row_text = " | ".join([cell.text.strip() for cell in row.cells if cell.text.strip()])
                if row_text:
                    text_content.append(row_text)
    except Exception as e:
        print(f"   [ERROR DOCX] {docx_path}: {e}")
    return "\n".join(text_content)

def extract_text_from_table(table_path: str) -> str:
    """Converts CSV and XLSX spreadsheet rows into plain text key-value lines."""
    text_content = []
    try:
        if table_path.endswith('.csv'):
            df = pd.read_csv(table_path)
        else:
            df = pd.read_excel(table_path)

        df = df.dropna(how='all')
        columns = [str(c).strip() for c in df.columns]

        for idx, row in df.iterrows():
            row_items = [f"{col}: {str(val).strip()}" for col, val in zip(columns, row) if pd.notna(val)]
            if row_items:
                text_content.append(f"Row {idx+1}: " + " | ".join(row_items))
    except Exception as e:
        print(f"   [ERROR Table] {table_path}: {e}")
    return "\n".join(text_content)

print("Multimodal Parsers Defined Successfully.")

Multimodal Parsers Defined Successfully.


In [4]:
def chunk_text_with_metadata(text: str, metadata: dict, chunk_size: int = 500, overlap: int = 60) -> list:
    """Splits document text into overlapping chunks with prepended metadata headers."""
    words = text.split()
    if not words:
        return []

    chunks = []
    # Prepend structural header for RAG context retention
    header = f"Document: {metadata.get('title', 'Untitled')} | Category: {metadata.get('category', 'General')} | Source: {metadata.get('source_url', '')}\n\n"

    start = 0
    chunk_idx = 0
    while start < len(words):
        end = start + chunk_size
        chunk_words = words[start:end]
        chunk_body = " ".join(chunk_words)

        chunk_record = {
            "chunk_id": f"{metadata['doc_id']}_c{chunk_idx}",
            "doc_id": metadata['doc_id'],
            "title": metadata.get('title', ''),
            "category": metadata.get('category', ''),
            "file_type": metadata.get('file_type', ''),
            "source_url": metadata.get('source_url', ''),
            "chunk_text": header + chunk_body,
            "char_count": len(chunk_body)
        }
        chunks.append(chunk_record)

        chunk_idx += 1
        start += (chunk_size - overlap)
        if start >= len(words) and len(chunks) == 0:
            break

    return chunks

print("Context-Aware Chunker Defined Successfully.")

Context-Aware Chunker Defined Successfully.


In [9]:
def run_stage_2_preprocessing():
    print("=== STARTING STAGE 2 PREPROCESSING & CHUNKING PIPELINE ===")

    if not os.path.exists(PATHS["metadata"]):
        print("Error: metadata.json missing.")
        return

    with open(PATHS["metadata"], "r", encoding="utf-8") as f:
        catalog = json.load(f)

    # Ensure extracted_text directory exists
    extracted_text_dir = os.path.join(PATHS["processed_dir"], "extracted_text")
    os.makedirs(extracted_text_dir, exist_ok=True)

    all_chunks = []
    processed_count = 0

    for idx, item in enumerate(catalog, 1):
        rel_path = item.get("local_path", "")
        full_path = os.path.join(PROJECT_ROOT, rel_path)
        file_type = item.get("file_type", "")

        if not os.path.exists(full_path):
            continue

        print(f"[{idx}/{len(catalog)}] Parsing ({file_type.upper()}) -> {item['title'][:40]}")
        extracted_text = ""

        if file_type == "html":
            with open(full_path, "r", encoding="utf-8") as f:
                extracted_text = f.read()
        elif file_type == "pdf":
            extracted_text = extract_text_from_pdf(full_path)
        elif file_type == "docx":
            extracted_text = extract_text_from_docx(full_path)
        elif file_type == "table":
            extracted_text = extract_text_from_table(full_path)

        if extracted_text.strip():
            # 1. Save physical text file for inspection/debugging
            txt_filename = f"{item['doc_id']}.txt"
            txt_path = os.path.join(extracted_text_dir, txt_filename)
            with open(txt_path, "w", encoding="utf-8") as tf:
                tf.write(extracted_text)

            # 2. Generate chunks for vector indexing
            chunks = chunk_text_with_metadata(extracted_text, item)
            all_chunks.extend(chunks)
            processed_count += 1

    # Save unified chunks dataset
    with open(PATHS["chunks_output"], "w", encoding="utf-8") as f:
        json.dump(all_chunks, f, indent=2)

    print("\n=== STAGE 2 PREPROCESSING COMPLETE ===")
    print(f"Processed Assets: {processed_count}/{len(catalog)}")
    print(f"Text files saved to: {extracted_text_dir}")
    print(f"Total Chunks Generated: {len(all_chunks)}")
    print(f"Chunk dataset saved to: {PATHS['chunks_output']}")

# Run Preprocessing Pipeline
run_stage_2_preprocessing()

=== STARTING STAGE 2 PREPROCESSING & CHUNKING PIPELINE ===
[1/410] Parsing (DOCX) -> Time Table
   [ERROR DOCX] E:\Rasengan\College-Knowledge-Search\data/raw/docs/General/timetable.docx: Package not found at 'E:\Rasengan\College-Knowledge-Search\data/raw/docs/General/timetable.docx'
[2/410] Parsing (PDF) -> Telephone Directory
[3/410] Parsing (DOCX) -> Networking Complaint
   [ERROR DOCX] E:\Rasengan\College-Knowledge-Search\data/raw/docs/General/viewform.docx: Package not found at 'E:\Rasengan\College-Knowledge-Search\data/raw/docs/General/viewform.docx'
[4/410] Parsing (PDF) -> Policy of Prevention of Sexual Harassmen
[5/410] Parsing (HTML) -> Indian Institute of Information Technolo
[6/410] Parsing (PDF) -> Learn more
[7/410] Parsing (PDF) -> Learn more
[8/410] Parsing (DOCX) -> Networking Complaint
   [ERROR DOCX] E:\Rasengan\College-Knowledge-Search\data/raw/docs/Institute/viewform.docx: Package not found at 'E:\Rasengan\College-Knowledge-Search\data/raw/docs/Institute/viewform.do

In [10]:
if os.path.exists(PATHS["chunks_output"]):
    with open(PATHS["chunks_output"], "r", encoding="utf-8") as f:
        chunks = json.load(f)

    print(f"Total Chunks Ingested: {len(chunks)}\n")
    print("Sample Chunk Payload:")
    print(json.dumps(chunks[0], indent=2))
else:
    print("chunks.json not generated yet. Run Cell 5 first.")

Total Chunks Ingested: 2353

Sample Chunk Payload:
{
  "chunk_id": "pdf_d34d247be2_c0",
  "doc_id": "pdf_d34d247be2",
  "title": "Telephone Directory",
  "category": "General",
  "file_type": "pdf",
  "source_url": "https://iiitn.ac.in/media/fb94fbf05b774a7e80dfef4a125c4dcb.pdf",
  "chunk_text": "Document: Telephone Directory | Category: General | Source: https://iiitn.ac.in/media/fb94fbf05b774a7e80dfef4a125c4dcb.pdf\n\n--- Page 1 --- sameer \u091f\u0947\u0932(cid:547)\u095e\u094b\u0928 \u0921\u093e\u0907\u0930\u0947(cid:200)\u091f\u0930(cid:547) / Directory (cid:272)(cid:871)\u092e\u0915 \u0938\u0902(cid:201)\u092f\u093e \u0928\u093e\u092e Name \u092a\u0926 Designation \u092e\u094b\u092c\u093e\u0907\u0932 \u0928\u0902\u092c\u0930 \u0907\u0902\u091f\u0930\u0915\u0949\u092e \u0928\u0902\u092c\u0930 \u0908\u092e\u0947\u0932 \u0906\u0908\u0921\u0940 Email- ID Sr No. Mobile Number Intercom No. 1 (cid:292)\u094b (\u0921\u0949) (cid:292)\u0947\u092e\u0932\u093e\u0932 \u092a\u091f\u0947\u0932